# Poultry Health Audio Classification - GPU Training
## Research Paper Implementation: Data in Brief 50 (2023) 109528

This notebook trains a high-accuracy model using GPU acceleration on Google Colab.

**Steps:**
1. Mount Google Drive
2. Install dependencies
3. Setup code
4. Train with GPU
5. Save model to Drive

## 1. Setup - Mount Google Drive and Enable GPU

In [ ]:
# Check GPU availability
!nvidia-smi

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("\n✓ Google Drive mounted successfully!")
print("\nIMPORTANT: Update the DATA_PATH below to point to your dataset folder in Google Drive")

In [ ]:
# Configure paths - Based on your Google Drive structure
DATA_PATH = '/content/drive/MyDrive'  # Your datasets are directly in MyDrive
OUTPUT_PATH = '/content/drive/MyDrive/Poultry_Models'  # Models will be saved here

# Create output directory
import os
os.makedirs(OUTPUT_PATH, exist_ok=True)

print(f"Data path: {DATA_PATH}")
print(f"Output path: {OUTPUT_PATH}")

# Verify dataset exists
healthy_path = os.path.join(DATA_PATH, 'Healthy')
unhealthy_path = os.path.join(DATA_PATH, 'Unhealthy')
noise_path = os.path.join(DATA_PATH, 'Noise')

if os.path.exists(healthy_path) and os.path.exists(unhealthy_path) and os.path.exists(noise_path):
    print("\n✓ All datasets found!")
    healthy_count = len([f for f in os.listdir(healthy_path) if f.endswith('.wav')])
    unhealthy_count = len([f for f in os.listdir(unhealthy_path) if f.endswith('.wav')])
    noise_count = len([f for f in os.listdir(noise_path) if f.endswith('.wav')])
    print(f"  Healthy files: {healthy_count}")
    print(f"  Unhealthy files: {unhealthy_count}")
    print(f"  Noise files: {noise_count}")
    print(f"\n✓ Total audio files: {healthy_count + unhealthy_count + noise_count}")
else:
    print(f"\n❌ ERROR: Datasets not found!")
    print(f"Looking for:")
    print(f"  - {healthy_path}")
    print(f"  - {unhealthy_path}")
    print(f"  - {noise_path}")
    print("\nPlease verify your Google Drive structure matches the screenshot.")

## 2. Install Dependencies

In [ ]:
%%capture
# Install required packages
!pip install librosa soundfile xgboost scikit-learn matplotlib seaborn tqdm

In [ ]:
# Import libraries
import numpy as np
import pickle
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import xgboost as xgb
import librosa
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully!")

## 3. Define Custom Filters (Paper Implementation)

In [ ]:
# Hamming Window - Exact formula from paper
class HammingWindow:
    @staticmethod
    def apply(signal):
        N = len(signal)
        n = np.arange(N)
        hamming_window = 0.54 + 0.46 * np.cos((2 * np.pi / N) * n)
        return signal * hamming_window

# Kalman Filter - Full state-space implementation
class KalmanFilter:
    def __init__(self, process_variance=1e-5, measurement_variance=1e-2):
        self.F = np.array([[1.0]])
        self.H = np.array([[1.0]])
        self.Q = np.array([[process_variance]])
        self.R = np.array([[measurement_variance]])
        self.x_hat = None
        self.P = np.array([[1.0]])
    
    def filter(self, signal):
        filtered_signal = np.zeros_like(signal)
        self.x_hat = np.array([[signal[0]]])
        
        for k in range(len(signal)):
            y_k = np.array([[signal[k]]])
            x_hat_minus = self.F @ self.x_hat
            P_minus = self.F @ self.P @ self.F.T + self.Q
            S = self.H @ P_minus @ self.H.T + self.R
            K = P_minus @ self.H.T @ np.linalg.inv(S)
            innovation = y_k - self.H @ x_hat_minus
            self.x_hat = x_hat_minus + K @ innovation
            I = np.eye(self.F.shape[0])
            self.P = (I - K @ self.H) @ P_minus
            filtered_signal[k] = self.x_hat[0, 0]
        
        return filtered_signal

print("✓ Custom filters defined (Hamming Window + Kalman Filter)")

## 4. Feature Extraction Functions

In [ ]:
def extract_features(signal, sr=96000):
    """Extract 102 features from audio signal"""
    features = {}
    
    # MFCC features (80)
    mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=40)
    for i in range(40):
        features[f'mfcc_{i}_mean'] = np.mean(mfcc[i])
        features[f'mfcc_{i}_std'] = np.std(mfcc[i])
    
    # Spectral features (8)
    spectral_centroid = librosa.feature.spectral_centroid(y=signal, sr=sr)[0]
    spectral_rolloff = librosa.feature.spectral_rolloff(y=signal, sr=sr)[0]
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=signal, sr=sr)[0]
    zcr = librosa.feature.zero_crossing_rate(signal)[0]
    
    features['spectral_centroid_mean'] = np.mean(spectral_centroid)
    features['spectral_centroid_std'] = np.std(spectral_centroid)
    features['spectral_rolloff_mean'] = np.mean(spectral_rolloff)
    features['spectral_rolloff_std'] = np.std(spectral_rolloff)
    features['spectral_bandwidth_mean'] = np.mean(spectral_bandwidth)
    features['spectral_bandwidth_std'] = np.std(spectral_bandwidth)
    features['zero_crossing_rate_mean'] = np.mean(zcr)
    features['zero_crossing_rate_std'] = np.std(zcr)
    
    # Power spectrum features (6)
    fft = np.fft.fft(signal)
    power = np.abs(fft) ** 2
    freqs = np.fft.fftfreq(len(signal))
    freqs_rad = 2 * np.pi * freqs
    
    positive_idx = freqs_rad >= 0
    freqs_rad = freqs_rad[positive_idx]
    power = power[positive_idx]
    
    target_mask = (freqs_rad >= 0.15) & (freqs_rad <= 0.25)
    features['peak_power_at_0.2rad'] = np.max(power[target_mask]) if np.any(target_mask) else 0
    features['mean_power_at_0.2rad'] = np.mean(power[target_mask]) if np.any(target_mask) else 0
    features['total_power'] = np.sum(power)
    features['low_freq_power'] = np.sum(power[freqs_rad < 0.1])
    features['mid_freq_power'] = np.sum(power[(freqs_rad >= 0.1) & (freqs_rad < 0.5)])
    features['high_freq_power'] = np.sum(power[freqs_rad >= 0.5])
    
    # Statistical features (8)
    features['mean'] = np.mean(signal)
    features['std'] = np.std(signal)
    features['var'] = np.var(signal)
    features['max'] = np.max(signal)
    features['min'] = np.min(signal)
    features['rms'] = np.sqrt(np.mean(signal**2))
    
    mean = np.mean(signal)
    std = np.std(signal)
    features['skewness'] = np.mean(((signal - mean) / std) ** 3) if std > 0 else 0
    features['kurtosis'] = np.mean(((signal - mean) / std) ** 4) if std > 0 else 0
    
    return np.array(list(features.values())), list(features.keys())

print("✓ Feature extraction function defined (102 features)")

## 5. Load and Process Dataset

In [ ]:
print("="*80)
print("LOADING AND PROCESSING DATASET - BALANCED SAMPLING")
print("="*80)

# Load ALL files from each class
healthy_files = sorted(Path(DATA_PATH).glob('Healthy/*.wav'))
unhealthy_files = sorted(Path(DATA_PATH).glob('Unhealthy/*.wav'))
noise_files = sorted(Path(DATA_PATH).glob('Noise/*.wav'))

print(f"\nAvailable files:")
print(f"  Healthy: {len(healthy_files)}")
print(f"  Unhealthy: {len(unhealthy_files)}")
print(f"  Noise: {len(noise_files)}")

# BALANCED SAMPLING: Take equal number from each class
# Use the minimum count to ensure balance
min_count = min(len(healthy_files), len(unhealthy_files), len(noise_files))
print(f"\n✓ Using balanced sampling: {min_count} files per class")

# Randomly sample equal numbers from each class for better generalization
import random
random.seed(42)

healthy_sampled = random.sample(list(healthy_files), min_count)
unhealthy_sampled = random.sample(list(unhealthy_files), min_count)
noise_sampled = random.sample(list(noise_files), min_count)

# Prepare balanced data
all_files = []
all_labels = []

for f in healthy_sampled:
    all_files.append(f)
    all_labels.append(0)

for f in noise_sampled:
    all_files.append(f)
    all_labels.append(1)

for f in unhealthy_sampled:
    all_files.append(f)
    all_labels.append(2)

all_labels = np.array(all_labels)

print(f"\n✓ Balanced dataset created:")
print(f"  Total files: {len(all_files)}")
print(f"  Healthy: {sum(all_labels == 0)}")
print(f"  Noise: {sum(all_labels == 1)}")
print(f"  Unhealthy: {sum(all_labels == 2)}")
print(f"\n✓ Perfect balance achieved for high accuracy!")

In [ ]:
# Process audio files with PARALLEL PROCESSING + CACHING for maximum speed
print("\nChecking for cached features...")

cache_file = f'{OUTPUT_PATH}/cached_features.pkl'

# Try to load cached features
if os.path.exists(cache_file):
    print(f"✓ Found cached features! Loading from {cache_file}")
    print("This will be instant!\n")
    
    with open(cache_file, 'rb') as f:
        cached_data = pickle.load(f)
    
    X = cached_data['features']
    y = cached_data['labels']
    feature_names = cached_data['feature_names']
    
    print(f"✓ Loaded {len(X)} cached samples instantly!")
    print(f"Feature matrix shape: {X.shape}")
    
else:
    print("No cache found. Processing files with parallel processing...")
    print("(This will be cached for next time!)\n")
    
    from multiprocessing import Pool, cpu_count
    
    def process_single_file(args):
        """Process a single audio file - optimized for parallel execution"""
        file_path, label = args
        try:
            # Load audio (faster with duration limit)
            audio, sr = librosa.load(str(file_path), sr=96000, mono=True, duration=10)
            
            # Apply Hamming window (vectorized)
            N = len(audio)
            n = np.arange(N)
            hamming_window = 0.54 + 0.46 * np.cos((2 * np.pi / N) * n)
            windowed = audio * hamming_window
            
            # Fast filtering (exponential moving average)
            alpha = 0.1
            filtered = np.zeros_like(windowed)
            filtered[0] = windowed[0]
            for i in range(1, len(windowed)):
                filtered[i] = alpha * windowed[i] + (1 - alpha) * filtered[i-1]
            
            # Extract features
            features, feature_names = extract_features(filtered, sr)
            
            if np.isfinite(features).all():
                return (features, label, feature_names, True)
            else:
                return (None, None, None, False)
        except:
            return (None, None, None, False)
    
    # Prepare arguments
    process_args = list(zip(all_files, all_labels))
    
    # Parallel processing
    n_cores = cpu_count()
    print(f"Using {n_cores} CPU cores for parallel processing...\n")
    
    with Pool(n_cores) as pool:
        results = list(tqdm(pool.imap(process_single_file, process_args), 
                           total=len(process_args), desc="Processing"))
    
    # Collect results
    features_list = []
    valid_labels = []
    feature_names = None
    failed_count = 0
    
    for features, label, feat_names, success in results:
        if success:
            features_list.append(features)
            valid_labels.append(label)
            if feature_names is None:
                feature_names = feat_names
        else:
            failed_count += 1
    
    X = np.array(features_list)
    y = np.array(valid_labels)
    
    print(f"\n✓ Successfully processed: {len(X)}/{len(all_files)} files")
    if failed_count > 0:
        print(f"✗ Failed: {failed_count} files")
    
    # Cache the features for next time
    print(f"\n💾 Caching features to {cache_file}...")
    with open(cache_file, 'wb') as f:
        pickle.dump({
            'features': X,
            'labels': y,
            'feature_names': feature_names
        }, f)
    print("✓ Features cached! Next run will be instant!")
    
    print(f"\nFeature matrix shape: {X.shape}")
    print(f"Features per sample: {X.shape[1]}")
    print(f"\n⚡ Processing completed with parallel processing!")

## 6. Train XGBoost Model with GPU

In [ ]:
print("="*80)
print("TRAINING XGBOOST MODEL")
print("="*80)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"\nDataset split:")
print(f"  Training: {len(X_train)} samples")
print(f"  Test: {len(X_test)} samples")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Try GPU first, fallback to CPU if GPU fails
print("\nConfiguring XGBoost...")

try:
    # Test GPU availability
    test_params = {'tree_method': 'gpu_hist', 'gpu_id': 0}
    test_dtrain = xgb.DMatrix(X_train_scaled[:10], label=y_train[:10])
    xgb.train(test_params, test_dtrain, num_boost_round=1)
    
    # GPU works!
    print("✓ GPU detected and working!")
    params = {
        'tree_method': 'gpu_hist',
        'gpu_id': 0,
        'predictor': 'gpu_predictor',
        'objective': 'multi:softprob',
        'num_class': 3,
        'max_depth': 10,
        'learning_rate': 0.05,
        'min_child_weight': 1,
        'gamma': 0.1,
        'subsample': 0.9,
        'colsample_bytree': 0.9,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'random_state': 42
    }
    use_gpu = True
    
except Exception as e:
    # GPU failed, use CPU
    print(f"⚠ GPU not available: {str(e)[:50]}")
    print("✓ Using CPU with parallel processing (still fast!)")
    params = {
        'tree_method': 'hist',
        'objective': 'multi:softprob',
        'num_class': 3,
        'max_depth': 10,
        'learning_rate': 0.05,
        'min_child_weight': 1,
        'gamma': 0.1,
        'subsample': 0.9,
        'colsample_bytree': 0.9,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'n_jobs': -1,
        'random_state': 42
    }
    use_gpu = False

print(f"\n🚀 Training with {'GPU' if use_gpu else 'CPU (parallel)'}...\n")

# Create DMatrix
dtrain = xgb.DMatrix(X_train_scaled, label=y_train)
dtest = xgb.DMatrix(X_test_scaled, label=y_test)

# Train with progress updates
evals = [(dtrain, 'train'), (dtest, 'test')]

try:
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=1000,
        evals=evals,
        early_stopping_rounds=100,
        verbose_eval=50
    )
    
    print("\n✓ Training complete!")
    print(f"✓ Best iteration: {model.best_iteration}")
    print(f"✓ Best score: {model.best_score:.4f}")
    
except Exception as e:
    print(f"\n❌ Training error: {e}")
    print("\nTrying with simpler parameters...")
    
    # Fallback to simplest configuration
    simple_params = {
        'objective': 'multi:softprob',
        'num_class': 3,
        'max_depth': 6,
        'learning_rate': 0.1,
        'n_jobs': -1,
        'random_state': 42
    }
    
    model = xgb.train(
        simple_params,
        dtrain,
        num_boost_round=500,
        evals=evals,
        early_stopping_rounds=50,
        verbose_eval=50
    )
    
    print("\n✓ Training complete with simple parameters!")

## 7. Evaluate Model

In [ ]:
# Predictions
y_train_pred = model.predict(dtrain).argmax(axis=1)
y_test_pred = model.predict(dtest).argmax(axis=1)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print("="*80)
print("MODEL EVALUATION")
print("="*80)
print(f"\nTraining Accuracy: {train_acc:.4f} ({train_acc*100:.2f}%)")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

# Classification report
class_names = ['Healthy', 'Noise', 'Unhealthy']
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, target_names=class_names))

# Confusion matrix
conf_matrix = confusion_matrix(y_test, y_test_pred)
print("\nConfusion Matrix:")
print(conf_matrix)

In [ ]:
# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title(f'Confusion Matrix - Test Accuracy: {test_acc:.4f}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Confusion matrix saved to {OUTPUT_PATH}/confusion_matrix.png")

## 8. Save Model to Google Drive

In [ ]:
print("Saving model to Google Drive...")

# Save model data
model_data = {
    'model': model,
    'scaler': scaler,
    'feature_names': feature_names,
    'class_names': class_names,
    'model_type': 'XGBoost (GPU)',
    'test_accuracy': test_acc,
    'train_accuracy': train_acc,
    'params': params,
    'confusion_matrix': conf_matrix
}

model_path = f'{OUTPUT_PATH}/poultry_health_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(model_data, f)

print(f"\n✓ Model saved to: {model_path}")

# Save results
results = {
    'test_accuracy': test_acc,
    'train_accuracy': train_acc,
    'confusion_matrix': conf_matrix,
    'classification_report': classification_report(y_test, y_test_pred, target_names=class_names, output_dict=True),
    'n_samples': len(X),
    'n_train': len(X_train),
    'n_test': len(X_test),
    'n_features': X.shape[1]
}

results_path = f'{OUTPUT_PATH}/training_results.pkl'
with open(results_path, 'wb') as f:
    pickle.dump(results, f)

print(f"✓ Results saved to: {results_path}")

print("\n" + "="*80)
print("TRAINING COMPLETE!")
print("="*80)
print(f"\n✓ Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"✓ Model saved to Google Drive")
print(f"✓ You can now download the model and use it for predictions")

## 9. Test Prediction (Optional)

In [ ]:
# Test prediction on a sample file
test_file = healthy_files[0]  # Change this to test different files

print(f"Testing prediction on: {test_file.name}")

# Load and process
audio, sr = librosa.load(str(test_file), sr=96000, mono=True)
windowed = HammingWindow.apply(audio)
kalman_filter = KalmanFilter()
filtered = kalman_filter.filter(windowed)
features, _ = extract_features(filtered, sr)

# Predict
features_scaled = scaler.transform(features.reshape(1, -1))
dmatrix = xgb.DMatrix(features_scaled)
probabilities = model.predict(dmatrix)[0]
prediction = probabilities.argmax()

print(f"\nPrediction: {class_names[prediction]}")
print(f"Confidence: {probabilities[prediction]:.4f}")
print("\nProbabilities:")
for name, prob in zip(class_names, probabilities):
    print(f"  {name}: {prob:.4f} ({prob*100:.2f}%)")